# Ишка: дообучение Qwen3-4B в Google Colab

**Перед запуском:** меню *Среда выполнения → Сменить среду выполнения → T4 GPU*. Затем запускай ячейки по порядку (Shift+Enter).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Код проекта
Вариант А — из GitHub (замени ссылку на свою). Вариант Б — загрузи `ishka.zip` через панель *Файлы* слева и раскомментируй вторую строку.

In [ ]:
!git clone https://github.com/ВАШ_НИК/ishka-assistant.git project
# !unzip -q ishka.zip -d project
%cd project

## 2. Библиотеки

In [ ]:
!pip install -q unsloth openai

## 3. Датасет: собрать и проверить

In [ ]:
!python db/build_db.py && python gen_dataset.py
!python validate_dataset.py data/train.jsonl | tail -2
!python validate_dataset.py data/eval.jsonl | tail -2

## 4. Точка отсчёта: базовая модель без дообучения (можно пропустить)

In [ ]:
!python evaluate.py --model unsloth/Qwen3-4B-Instruct-2507 --limit 50 | tail -4

## 5. Обучение
Прогресс видно по `loss`: должен уменьшаться. Если не хватает памяти — в `train_qlora.py` уменьши `per_device_train_batch_size` до 1 или `MAX_LEN` до 2048.

In [ ]:
!python train_qlora.py

## 6. Оценка дообученной модели

In [ ]:
!python evaluate.py --model out/lora --limit 100 | tail -15

## 7. Сохранить LoRA на Google Диск (чтобы не потерять при отключении Colab)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/ishka && cp -r out/lora /content/drive/MyDrive/ishka/

## 8. Экспорт в GGUF (для запуска на ноутбуке/сервере через llama.cpp или Ollama)
Для загрузки на Hugging Face: слева *Секреты* → добавь `HF_TOKEN` (токен с правом write с huggingface.co/settings/tokens) и раскомментируй строку.

In [ ]:
!python export.py --no-merged
# import os; from google.colab import userdata; os.environ['HF_TOKEN']=userdata.get('HF_TOKEN')
# !python export.py --no-merged --hf ВАШ_НИК/ishka
!ls -lh out/gguf

In [ ]:
!cp out/gguf/*.gguf /content/drive/MyDrive/ishka/   # копия на Диск, оттуда скачаешь на компьютер